# Exploration sim_v1

Notebook d'exploration uniquement : connexion a la base, lecture des tables/vues
via `table_view` / `p_table_view`. Aucune logique metier ici.


## 1. Setup


In [ ]:
from pathlib import Path
import sys

# Racine release (dossier de ce notebook)
ROOT = Path.cwd().resolve()
if not (ROOT / "pipeline").is_dir():
    # si le kernel a un autre cwd, remonter depuis le fichier
    ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipeline.paths import Paths, release_root
from src.pipeline.connection import PipelineFactory
from src.pipeline.engine import ConnectionPipeline

paths = Paths(ROOT).ensure()
print("ROOT     :", paths.root)
print("DB       :", paths.main_db, "exists=", paths.main_db.exists())
print("pipeline :", paths.pipeline)


## 2. Connexion pipeline


In [ ]:
# Connexion lecture/ecriture sur la base principale + YAML pipeline/
cp = PipelineFactory(paths).open(read_only=False)
print("project_dir :", cp.project_dir)
print("objets pipeline YAML :", len(cp.pipeline))


## 3. Relations dans la base


In [ ]:
import duckdb

def list_relations(cp, like: str | None = None):
    """Liste tables et vues de la base ouverte."""
    q = """
        SELECT table_name, table_type
        FROM information_schema.tables
        WHERE table_schema = current_schema()
        ORDER BY table_type, table_name
    """
    df = cp.con.execute(q).df()
    if like:
        df = df[df["table_name"].str.contains(like, case=False, na=False)]
    return df

rels = list_relations(cp)
display(rels)
print(f"{len(rels)} relations")


## 4. Filtrer les objets sim_v1

Les tables/vues v1 sont en general prefixees `t_v1_`, `v_v1_`, ou liees aux features hotel.


In [ ]:
v1_rels = list_relations(cp, like=r"v1|hotel_param|pilot|scope")
display(v1_rels)


## 5. `table_view` — relation deja materialisee


In [ ]:
# Exemple : parametres hotel (si la table existe)
name = "t_hotel_params"
if cp.relation_exists(name):
    df = cp.table_view(name).df()
    display(df.head(20))
    print(df.shape, list(df.columns)[:20])
else:
    print(f"{name} absente — lancer le pipeline sim_v1 ou p_table_view ci-dessous")


## 6. `p_table_view` — construit avec prerequis automatiques

`process_with_requires` charge les dependances YAML puis renvoie la relation.


In [ ]:
# Construit v_hotel_params (et ses requires) si besoin
try:
    df = cp.p_table_view("v_hotel_params").df()
    display(df)
    print(df.shape)
except Exception as exc:
    print("p_table_view v_hotel_params :", exc)


## 7. Resultats LOO sim_v1


In [ ]:
for name in ("t_v1_loo_results", "v_v1_loo_metrics", "t_v1_loo_hotels"):
    print("===", name, "===")
    if not cp.relation_exists(name):
        print("  (absente)")
        continue
    d = cp.table_view(name).df()
    display(d.head(30))
    print("  shape", d.shape)


## 8. Explorer une relation au choix


In [ ]:
# Modifier le nom pour inspecter n'importe quelle table/vue
NAME = "v_v1_prediction"  # peut necessiter v_loo_step (iteration)

if cp.relation_exists(NAME):
    try:
        display(cp.table_view(NAME).df().head(50))
    except Exception as exc:
        print("table_view:", exc)
else:
    try:
        display(cp.p_table_view(NAME).df().head(50))
    except Exception as exc:
        print("p_table_view:", exc)


## 9. Fermer la connexion


In [ ]:
cp.close()
print("connexion fermee")
